# EA3 — Procesamiento distribuido de datos con Apache Spark

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 23 |
| **Integrantes** | Manuela Tangarife Zapata |
| **Caso de estudio** | Wanderbricks |
| **Fecha de entrega** | domingo 28 de septiembre |
| **🎥 Enlace al video** | https://youtu.be/HIiqp8kLPog |


---
## 1. Contexto y problema

El caso de estudio de Wanderbricks permite analizar datos de reservas, propiedades, destinos y comportamiento de los usuarios en una plataforma de alojamiento.

El objetivo de esta evidencia es completar el procesamiento de los datos con Apache Spark, desde las capas de almacenamiento hasta una tabla de consumo. También se pretende evaluar el costo de una operación distribuida, aplicar una técnica de optimización y comparar los resultados antes y después de aplicarla.

La pregunta de negocio que escogí es: ¿Qué destinos generan mayor valor en reservas y qué relación existe entre las interacciones de los usuarios y las reservas realizadas?

Usaré las tablas de reservas, propiedades, destinos y eventos de navegación para responderla. El procesamiento distribuido permite combinar y agregar estos datos mediante Spark, aunque el rendimiento dependerá del volumen, la distribuición de los datos y el plan de ejecución.

---
## 2. Descripción de los datos

In [0]:
# Verificación del volumen con el que van a trabajar
TABLA_PRINCIPAL = "samples.tpch.lineitem"   # TODO: ajustar
TABLA_SECUNDARIA = "samples.tpch.orders"    # TODO: ajustar

for t in [TABLA_PRINCIPAL, TABLA_SECUNDARIA]:
    print(f"{t:35s} {spark.table(t).count():>12,} filas")

In [0]:
# Definición de la lista con los nombres de todas las tablas.
tablas_volumen = [
    "bookings_silver",
    "bookings_bronze",
    "clickstream_bronze",
    "clickstream_silver",
    "properties_bronze",
    "properties_silver",
    "destinations_bronze",
    "destinations_silver",
    "destinations_gold",
    "reviews_bronze",
    "reviews_silver",
    "payments_silver",
    "payments_bronze",
    "customer_support_logs_bronze",
    "customer_support_logs_silver",   
]

# Bucle para recorrer e inspeccionar el volumen
for nombre in tablas_volumen:
    tabla = f"bigdata_grupo23.wanderbricks.{nombre}"
    cantidad = spark.table(tabla).count()
    print(f"{nombre:35s}{cantidad:>12,} filas")

---
## 3. Decisiones de diseño y justificación

En esta evidencia conservo la arquitectura medallón utilizada en las evidencias anteriores:
- Bronze: almacenamiento de los datos originales.
- Silver: datos transformados y preparados para el análisis.
- Gold: información agregada y organizada para responder preguntas de negocio.

Para el análisis se relacionarán las reservas con las propiedades y los destinos. Además se resumirán los eventos de navegación por propiedad para estudiar las interacciones de los usuarios. 
La operación que elegí para la optimización será el cruce entre los resúmenes de reservas y los resúmenes de eventos por propiedad. Esta agregación previa evita generar una combinación de cada evento con cada reserva de una misma propiedad, lo que podría multiplicar innecesariamente los registros.

Como técnica de optimización, utilizare broadcast join. Lo que me permitira enviar una tabla pequeña a los nodos que porcesan la tabla más grande, evitando el intercambio y la redistribución de datos que pueden requerir un SortMergeJoin. La eficacia de esta ténica se comprobará mediante los planes de ejecución y los tiempos medidos.

---
## 4. Implementación
### 4.1 Herramienta de medición

In [0]:
import time, statistics

def medir(funcion, repeticiones=3, etiqueta=""):
    """Ejecuta la función varias veces y reporta la mediana."""
    tiempos = []
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)
    mediana = statistics.median(tiempos)
    print(f"{etiqueta:35s} mediana {mediana:6.2f}s   corridas {[round(t,2) for t in tiempos]}")
    return mediana

In [0]:
import time 
import statistics

# Funcion para bechmarking de consultas
def medir(funcion, repeticiones=3, etiqueta=""):
    tiempos = []
    
    # Interacciones de prueba
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        fin = time.time()

        tiempos.append(fin - inicio)
        # Calcula la mediana
        mediana = statistics.median(tiempos)

        print(f"\n{etiqueta}")
        print("Tiempos:", [round(t, 3) for t in tiempos])
        print(f"Mediana: {mediana:.3f} segundos")

        return mediana

La función permite ejecutar una operación tres veces y calcular la mediana de los tiempos. Antes de cada conjunto de mediciones se realizará una ejecución de calentamiento, que no se tendrá en cuenta en el resultadp.

Se medirá el tiempo de una acción que obligue a Spark a ejecutar la consulta en lugar de medir únicamente la definición de una transformación.

### 4.2 Capa bronce

In [0]:
# Consultar las tablas de bronce
bronze = [
    "bookings_bronze",
    "clickstream_bronze",
    "properties_bronze",
    "destinations_bronze",
    "customer_support_logs_bronze",
    "hosts_bronze",
    "payments_bronze",
    "reviews_bronze",
    "users_bronze"
]

# Recorre cada tabla Bronze
for nombre in bronze:
    tabla = f"bigdata_grupo23.wanderbricks.{nombre}"
    print(nombre, spark.table(tabla).count())

### 4.3 Capa plata — calidad de datos

Las tablas de plata ya se encuentran creadas. Para documentar su recorrido, se comparan los conteos de bronce y plata. Las diferencias permiten identidicar las tablas que requieren una revisión adicional.

En las reservas, propiedades, usuarios y anfitriones, los conteos de bronce y plata coinciden. En pagos y reseñas se observa una disminución de registros. En los registros de atención al cliente, la cantidad de filas aumenta porque la estructura original contiene un arreglo de mensajes que puede convertirse en varias filas.

In [0]:
# Comparar los registros de bronce y plata
pares = [
    ("bookings", "bookings_bronze", "bookings_silver"),
    ("clickstream", "clickstream_bronze", "clickstream_silver"),
    ("customer_support_logs",
     "customer_support_logs_bronze",
     "customer_support_logs_silver"),
    ("destinations", "destinations_bronze", "destinations_silver"),
    ("hosts", "hosts_bronze", "hosts_silver"),
    ("payments", "payments_bronze", "payments_silver"),
    ("properties", "properties_bronze", "properties_silver"),
    ("reviews", "reviews_bronze", "reviews_silver"),
    ("users", "users_bronze", "users_silver")
]

# Recorre cada tabla para auditar cambios de volumen entre capas
for nombre, tabla_bronze, tabla_silver in pares:
    # Cuenta filas en la capa Bronze
    bronce = spark.table(
        f"bigdata_grupo23.wanderbricks.{tabla_bronze}"
    ).count()
    # Lo mismo pero para la capa Plata
    plata = spark.table(
        f"bigdata_grupo23.wanderbricks.{tabla_silver}"
    ).count()

    # Calcula la variación de registros tras la limpieza
    diferencia = plata - bronce

    # Muestra los totales y la diferencia formateados
    print(
        f"{nombre:25s} "
        f"Bronce: {bronce:>9,} "
        f"Plata: {plata:>9,} "
        f"Diferencia: {diferencia:>8,}"
    )

In [0]:
from pyspark.sql import functions as F

bookings = spark.table(
    "bigdata_grupo23.wanderbricks.bookings_silver"
)

# Revisar valores nulos y duplicados en reservas
bookings.select(
    # Total de filas
    F.count("*").alias("total_registros"),
    #IDs unicos
    F.countDistinct("booking_id").alias("reservas_unicas"),
    # Conteos de IDs nulos
    F.sum(F.col("booking_id").isNull().cast("int")).alias(
        "ids_nulos"
    ),
    # Propiedades sin ID
    F.sum(F.col("property_id").isNull().cast("int")).alias(
        "propiedades_nulas"
    ),
    # Montos sin valor
    F.sum(F.col("total_amount").isNull().cast("int")).alias(
        "montos_nulos"
    )
).show()

In [0]:
clickstream = spark.table(
    "bigdata_grupo23.wanderbricks.clickstream_silver"
)

# Audita la presencia de valores nulos en columnas clave de clickstream
clickstream.select(
    F.count("*").alias("total_eventos"),
    F.sum(F.col("property_id").isNull().cast("int")).alias(
        "propiedades_nulas"
    ),
    F.sum(F.col("event").isNull().cast("int")).alias(
        "eventos_nulos"
    ),
    F.sum(F.col("timestamp").isNull().cast("int")).alias(
        "fechas_nulas"
    )
).show()

### 4.4 Transformaciones distribuidas

Para analizar la relación entre las reservas y las interacciones, primero agrupo los datos por propiedad. Después realizo el cruce de ambos resúmenes. Esta estrategia reduce la cantidad de filas que participan en el join.

También utilizo una función de ventana para ordenar los destinos según el valor total de las reservas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ruta = "bigdata_grupo23.wanderbricks"

bookings = spark.table(f"{ruta}.bookings_silver")
clickstream = spark.table(f"{ruta}.clickstream_silver")
properties = spark.table(f"{ruta}.properties_silver")
destinations = spark.table(f"{ruta}.destinations_silver")

# Resumen de reservas por propiedad
reservas_propiedad = (
    bookings
    .filter(F.col("status") != "revision")
    .groupBy("property_id")
    .agg(
        F.count("*").alias("total_reservas"),
        F.sum("total_amount").alias("valor_total_reservas")
    ) 
)

# Resumen de eventos por propiedad
eventos_propiedad = (
    clickstream
    .filter(F.col("property_id").isNotNull())
    .groupBy("property_id")
    .agg(
        F.count("*").alias("total_interacciones")
    )
)

#cruce de los resúmenes por propiedad
resumen_propiedades = (
    reservas_propiedad
    .join(eventos_propiedad, "property_id", "full")
    .fillna(0, subset=[
        "total_reservas",
        "valor_total_reservas",
        "total_interacciones"
    ])
    .join(properties, "property_id", "left")
    .join(destinations, "destination_id", "left")
    .select(
        "property_id",
        "title",
        "destination",
        "country",
        "total_reservas",
        "valor_total_reservas",
        "total_interacciones"
    )
)

display(resumen_propiedades)

In [0]:
# calcular el ranking de destinos por valor de reservas
ventana = Window.orderBy(
    F.desc("Valor_total_reservas")
)

ranking_destinos = (
    spark.table(f"{ruta}.destinations_gold")
    .withColumn(
        "ranking",
        F.row_number() .over(ventana)
    )
)

display(ranking_destinos)

### 4.5 Medición base

La cosnsulta base combina los resúmenes de reservas y eventos por propiedad, sin aplicar una optimización manual. Utilizo count() para forzar la ejecución de la consulta y poder medir el tiempo real de procesamiento

In [0]:
# crear la consulta base sin optimización manual
def consulta_base():
    # Agrupa y resume reservas en plata filtrando estados en revisión
    reservas = (
        spark.table(f"{ruta}.bookings_silver")
        .filter(F.col("status") != "revision")
        .groupBy("property_id")
        .agg(
            F.count("*").alias("total_reservas"),
            F.sum("total_amount").alias("valor_total_reservas")
        )
    )

    # Agrupa y cuenta eventos web en plata excluyendo propiedades nulas
    eventos = (
        spark.table(f"{ruta}.clickstream_silver")
        .filter(F.col("property_id").isNotNull())
        .groupBy("property_id")
        .agg(
            F.count("*").alias("total_interacciones")
        )
    )

    # Cruza reservas y eventos mediante FULL JOIN
    resultado = reservas.join( 
        eventos,
        on="property_id",
        how="full"            
    )

    return resultado

In [0]:
df_base = consulta_base()

#mostrar el plan de ejecución inicial
df_base.explain(mode="formatted")

In [0]:
# Ejecución de calentamiento: no se incluye en las mediciones
df_base.count()

# Tres mediciones de la consulta base 
t_base = medir(
    lambda: consulta_base().count(),
    repeticiones=3,
    etiqueta="Consulta base"
)

### 4.6 Optimización

La técnica que seleccione es broadcast join. Se aplicará al resumen de eventos por propiedad, que se distribuirá a los nodos que procesan el resumen de reservas.

Con esta técnica puedo reducir el intercambio de datos entre particiones cuando una de las tablas que se cruzan es suficientemente pequeña. Sin embargo su efectividad depende del tamaño real de los datos y de las decisiones que tome el optimizador de Spark.

In [0]:
def consulta_optimizada():
    pass

t_opt = medir(consulta_optimizada, etiqueta="Optimizada")
print(f"\nMejora: {(1 - t_opt/t_base)*100:.1f}%")

In [0]:
from pyspark.sql.functions import broadcast

# Crear la consulta optimizada con broadcast join
def consulta_optimizada():
    reservas = (
        spark.table(f"{ruta}.bookings_silver")
        .filter(F.col("status") != "revision")
        .groupBy("property_id")
        .agg(
            F.count("*").alias("total_reservas"),
            F.sum("total_amount").alias("valor_total_reservas")
        )
    )

    # Agrupa y cuenta eventos web en plata excluyendo propiedades nulas
    eventos = (
        spark.table(f"{ruta}.clickstream_silver")
        .filter(F.col("property_id").isNotNull())
        .groupBy("property_id")
        .agg(
            F.count("*").alias("total_interacciones")
        )
    )

    # Aplica broadcast a eventos para enviarla a los nodos y evitar shuffle    
    resultado = reservas.join(
        broadcast(eventos),
        on="property_id",
        how="full"
    )

    return resultado

In [0]:
df_opt = consulta_optimizada()

# Mostrar el plan de ejecución optimizado
df_opt.explain(mode="formatted")

In [0]:
# Ejecución de calentamiento
df_opt.count()

# Tres mediciones de la consulta optimizada
t_opt = medir(
    lambda: consulta_optimizada().count(),
    repeticiones=3,
    etiqueta="Consulta optimizada"
)

# Comparar las medianas 
if t_base > 0:
    mejora = (1 - t_opt / t_base) * 100
    print(f"\nMediana base: {t_base:.3f} segundos")
    print(f"Mediana optimizada: {t_opt:.3f} segundos")
    print(f"Variación del tiempo: {mejora:.1f}%")

### 4.7 Capa oro y consumo

In [0]:
# Preparar los datos para la capa oro
reservas = (
    spark.table(f"{ruta}.bookings_silver")
    .filter(F.col("status") != "revision")
    .groupBy("property_id")
    .agg(
        F.count("*").alias("total_reservas"),
        F.sum("total_amount").alias("valor_total_reservas")
    )
)

eventos = (
    spark.table(f"{ruta}.clickstream_silver")
    .filter(F.col("property_id").isNotNull())
    .groupBy("property_id")
    .agg(
        F.count("*").alias("total_interacciones")
    )
)

propiedades = spark.table(f"{ruta}.properties_silver")
destinos = spark.table(f"{ruta}.destinations_silver")

# Relacionar reservas, eventos, propiedades y destinos
datos = (
    propiedades
    .join(reservas, "property_id", "left")
    .join(eventos, "property_id", "left")
    .join(destinos, "destination_id", "left")
    .fillna(0, subset=[
        "total_reservas",
        "valor_total_reservas",
        "total_interacciones"
    ])
)

# Agreagr la informacion por destino
oro = (
    datos
    .filter(F.col("destination").isNotNull())
    .groupBy("destination", "country",)
    .agg(
        F.sum("total_reservas").cast("long").alias(
            "total_reservas"
        ),
        F.sum("valor_total_reservas").alias(
            "valor_total_reservas"
        ),
        F.sum("total_interacciones").cast("long").alias(
            "total_interacciones"
        )
    )
    .withColumn(
        "valor_promedio_reserva",
        F.when(
            F.col("total_reservas") > 0,
            F.col("valor_total_reservas") /
            F.col("total_reservas")
        ).otherwise(0)
    )
)

# Crear un ranking por valor total de reservas
ventana = Window.orderBy(
    F.desc("valor_total_reservas"),
    F.asc("destination")
)

oro = (
    oro
    .withColumn("ranking", F.row_number().over(ventana))
    .withColumn("actualizado_en", F.current_timestamp())
    .select(
        "destination",
        "country",
        "total_reservas",
        "valor_total_reservas",
        "valor_promedio_reserva",
        "total_interacciones",
        "ranking",
        "actualizado_en"
    )
    .orderBy("ranking")
)

display(oro)

In [0]:
# Guardar la capa oro ocmo una tabla Delta
(
    oro.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{ruta}.ea3_destinations_gold")
)

In [0]:
# Consultar la tabla oro creada
oro_final = spark.table(f"{ruta}.ea3_destinations_gold")

print("Registros de la capa oro:", oro_final.count())
display(oro_final)

---
## 5. Resultados

| Pregunta del negocio | Respuesta obtenida |
|---|---|
| ¿Qué destino tiene el mayor valor total de reserva? | Phuket, Tailandia es el destino con mayor valor total de reservas, con aproximadamente 4.058.199,55.|
| ¿Qué destino registra más reservas? | Phuket, Tailandia, con 7.292 reservas. |
| ¿Qué destino registra más interacciones de usuarios? | Phuket, Tailandia, con 9.852 interacciones. |

In [0]:
# Mostrar los tres destinos con mayor valor de reservas
display(
    spark.table(f"{ruta}.ea3_destinations_gold")
    .orderBy(F.desc("valor_total_reservas"))
    .limit(3)
)

## Opción A - Dashboard sobre la capa oro

Como actividad complementaria, realice un dashboard a partir de la tabla ea3_destinations_gold, con el próposito de facilitar la interpretación de la información de la capa oro y responder diferentes aspectos de la pregunta de negocio.

El dashboard contiene tres visualizaciones: Top 10 destinos por reservas, que permite identificar los destinos con mayor demanda, Top 10 destinos por valor de reservas, que permite comparar el valor económico generado por los destinos, y Top 10 destinos por interacciones, que permite identificar los destinos que presentan mayor nivel de interacción de los usuarios.

En conjunto las tres visualizaciones permiten analizar la demanda, el valor económico y el interés de los usuarios a partir de los datos procesados en la capa oro.

## Consulta reservas

Pregunta de negocio: ¿Qué destinos concentran la mayor cantidad de reservas?

Considero que la pregunta importa por que permite identificar dónde existe mayor demanda y orientar allí la disponibilidad y las estrategias comerciales.

In [0]:
%sql
SELECT
    destination,
    total_reservas
FROM bigdata_grupo23.wanderbricks.ea3_destinations_gold
ORDER BY total_reservas DESC
LIMIT 10

Databricks visualization. Run in Databricks to view.

## Consulta valor de reservas

Pregunta de negocio: ¿Qué destinos generan el mayor valor económico en reservas?

Esta pregunta ayuda a identificar los destinos que representan una mayor contribución económica para el negocio.

In [0]:
%sql
SELECT 
    destination,
    valor_total_reservas
FROM bigdata_grupo23.wanderbricks.ea3_destinations_gold
ORDER BY valor_total_reservas DESC
LIMIT 10

Databricks visualization. Run in Databricks to view.

## Consulta interacciones

Pregunta de negocio: ¿Qué destinos generan mayor interés entre los usuarios?

Con esta pregunta se puede conocer dónde existe mayor interacción y detectar destinos que pueden requerir acciones comerciales o de promoción.

In [0]:
%sql
SELECT 
    destination,
    total_interacciones
FROM bigdata_grupo23.wanderbricks.ea3_destinations_gold
ORDER BY total_interacciones DESC
LIMIT 10


Databricks visualization. Run in Databricks to view.

## Dashboard - Análisis de destinos

![image_1790558427978.png](./image_1790558427978.png "image_1790558427978.png")

### Solución Pregunta Orientadora del Curso

Una empresa que recibe datos de su portal web, facebook, instagram, tiktok y WhatsApp Business puede analizarlos mediante un paradisgma distribuido, como el procesamiento por lotes y cuando sea necesario, el procesamiento de flujos.
Le elección depende del volumen, la velocidad de llegada y el tipo de análisis que se requiera. Para almacenar y procesar los datos, se puede utilizar una arquitectura de lago de datos o lakehouse organizada en capas que permiran conservar los datos originales, transformarlos y preparar información para el consumo.

En el caso desarrollado durante el curso, databricks y apache spark permitieron trabajar con datos de reservas, propiedades y eventos de navegación mediante las capas Bronze, Silver y Gold. Una arquitectura similar podría integrar datos de redes sociales y del portal web, aplicando procesos de limpieza, estandarización y unión para obtener indicadores comunes. También sería necesario considerar la calidad de los datos, los permisos de acceso y el rendimiento de las operaciones distribuidas. La experiencia con Wanderbricks muestra que no basta con construir una consulta, también es importante revisar su plan de ejecución y medir si las optimizaciones realmente aportan beneficios.

---
## 6. Conclusiones

En este proyecto logré construir y analizar una arquitectura de datos por capas para wanderbricks, permitiendo transformar y organizar la información desde la capa Bronce hasta la capa Oro. Durante el desarrollo identifique cambios en el volumen de los datos debido a procesos de limpieza y transformación, y aplique una estrategia de optimización mediante broadcast para mejorar el procesamiento de datos. Aunque se presentaron algunas dificultades técnicas durante la construcción y análisis, el proyecto me permitió comprender mejor el funcionamiento de una arquitectura Lakehouse y la importancia de optimizar las consultas según las características de los datos. Finalmente esta experiencia me permitió reconocer comó herramientas como Spark, Delta lake y Databricks pueden utilizarse para integrar y analizar información proveniente de diferentes fuentes y apoyar la toma de decisiones basada en datos.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Manuela Tangarife Zapata | Toda la elaboración de la actividad | Justificación de la estructura de la capa oro, comparación de los planes de ejecución antes y después de la optimización, razón para sleccionar broadcast(), límites de esta optimización y respuesta a la pregunta orientadora. |


**Uso de asistentes de IA:** Utilice Copilot como herramienta de apoyo principalmente para consultar y aclarar aspectos relacionados con la sintaxis y estructura de algunas consultas y funciones de Spark.

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. Muestre el plan de ejecución antes y después de optimizar e indique qué cambió.
2. ¿Por qué esa técnica mejoró el tiempo en este caso concreto?
3. ¿En qué situación su optimización dejaría de servir o sería contraproducente?

---
## ✅ Antes de entregar

- [ ] El volumen mínimo está verificado y visible en la salida
- [ ] Las mediciones tienen tres corridas y se reporta la mediana
- [ ] Los planes de ejecución antes y después están en el notebook
- [ ] El video muestra ambos planes — es obligatorio en esta evidencia
- [ ] La capa oro responde las tres preguntas del negocio
- [ ] Todo confirmado en /ea3 y el HTML subido a Canvas